In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from configs.config_loader import Config
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)
notebook_config = Config("configs/app_notebook.yaml")


DOMAIN = "purchase"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name, new_config=notebook_config)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("bench") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("bench") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("bench") / "data_dir/domains" / "base"
BASE_CODE_DIR      = Path("bench") / "domains" / "base"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/purchase
生成数据目录：bench/data_dir/domains/purchase
生成代码目录：bench/domains/purchase
BASE 数据目录：bench/data_dir/domains/base


In [2]:
# 加载 BASE 域生成结果（作为 WORK 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()
with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")
print(f"BASE 表数量：{len(base_data)}")

加载完成
BASE 表数量：16


### 1. 生成 WORK Schema

In [ ]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", base_schema)

print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

### 2. 生成 Data Models

In [ ]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    models_prompt = f.read()

models_prompt = models_prompt.replace("{base_models}", base_models)
models_prompt = models_prompt.replace("{domain_schema}", domain_schema)

print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
models_response = ""
for chunk in llm.llm.stream(models_prompt):
    content = chunk.content
    models_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

models_code = extract_code_block(models_response)
is_valid, err = verify_python_syntax(models_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {err}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), models_code)
print(f"✓ Models 已保存到: {models_path}")

### 3. Schema/Model一致性检查

In [ ]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{models_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

### 4. 生成数据脚本（Faker，主流程）

In [ ]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 IT 域的外键依赖（不含 BASE 表），推导生成顺序
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"IT 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))

local_llm = LLMClient("gpt-5.4-0305-global", 128000, new_config=notebook_config)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

### 3.5 执行数据脚本，生成 db.json

In [5]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)

print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（WORK 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ WORK Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 29 张表
  - procurement_category: 50 条
  - supplier: 150 条
  - sourcing_project: 37 条
  - purchase_requisition: 150 条
  - requisition_item: 389 条
  - contract_item: 88 条
  - purchase_order_item: 199 条
  - purchase_change_request: 75 条
  - goods_receipt_item: 94 条
  - acceptance_record: 37 条
  - acceptance_item: 67 条
  - approval_flow_template: 15 条
  - approval_flow_node: 46 条
  - approval_instance: 75 条
  - approval_action: 124 条
  - procurement_permission_rule: 37 条
  - procurement_training_course: 12 条
  - procurement_training_record: 150 条
  - procurement_issue_ticket: 150 条
  - supplier_contact: 313 条
  - supplier_qualification: 420 条
  - supplier_category: 379 条
  - supplier_evaluation: 292 条
  - sourcing_supplier: 148 条
  - quotation: 75 条
  - purchase_contract: 37 条
  - purchase_order: 75 条
  - goods_receipt: 37 条
  - quotation_item: 198 条
✓ 外键完整性验证通过

✓ WORK Database 已保存到: bench/data_dir/domains/purchase/db.json


In [6]:
# Pydantic 模型验证：合并 BASE + WORK 数据后做完整校验
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from bench.domains.purchase.models import PurchaseDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = json.load(f)

merged_raw = {**base_data, **db_raw}

try:
    work_db = PurchaseDB.model_validate(merged_raw)
    print("✓ WorkDB 模型验证通过")
    print(work_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ WorkDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_procurement_categories': 50, 'num_suppliers': 150, 'num_supplier_contacts': 313, 'num_supplier_qualifications': 420, 'num_supplier_categories': 379, 'num_supplier_evaluations': 292, 'num_sourcing_projects': 37, 'num_sourcing_suppliers': 148, 'num_quotations': 75, 'num_quotation_items': 198, 'num_purchase_requisitions': 150, 'num_requisition_items': 389, 'num_purchase_contracts': 37, 'num_contract_items': 88, 'num_purchase_orders': 75, 'num_purchase_order_items': 199, 'num_purchase_change_requests': 75, 'num_goods_receipts': 37, 'num_goods_receipt_items': 94, 'num_acceptance_records': 37, 'num_acceptance_items': 67, 'num_approval_flow_templates': 15, 'num_approval_flow_nodes': 46, 'num_approval_instances': 75, 'num_approval_actions': 124, 'num_procurement_permission_rules': 37, 'num_procurement_training_courses': 12, 'num_procurement_training_records

### 4. 生成 WORK Tools

In [8]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# WORK 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
work_function_description = """""采购供应链": [
    {
      "title": "采购类目选择",
      "definition": "用户希望人工智能协助进行采购相关的分类和选择工作，包括供应商筛选、产品类别划分、采购方案比较、供应商评估标准制定，以及基于需求匹配最适合的采购类目和供应商推荐。"
    },
    {
      "title": "特殊采购申请",
      "definition": "用户需要为非常规采购需求创建申请文档，包括紧急采购、超预算采购、非标准供应商采购等特殊情况的申请表格生成，以及相关审批流程文档的自动化创建。"
    },
    {
      "title": "采购审批流程管理",
      "definition": "用户希望通过人工智能助手来优化和管理采购审批流程，包括创建采购申请、跟踪审批状态、自动分配审批人员、生成采购审批报告，以及对采购流程进行分析和优化建议。"
    },
    {
      "title": "采购变更管理",
      "definition": "用户希望对采购过程中的变更事项进行有效管理和跟踪，包括供应商变更、采购需求调整、合同条款修改、价格变动、交付时间调整等变更的申请、审批、执行和记录，确保采购活动的合规性和可控性。"
    },
    {
      "title": "采购方式选择",
      "definition": "用户需要人工智能协助分析和选择最适合的采购策略、供应商评估方法或采购流程，包括比较不同采购模式（如集中采购、分散采购、战略采购）的优劣势，评估供应商资质和报价方案，以及制定采购决策建议。"
    },
    {
      "title": "采购合同关联下单",
      "definition": "用户希望将采购合同与实际的采购订单进行关联管理，包括根据合同条款自动生成采购订单、跟踪合同执行进度、核对订单与合同的一致性，以及管理合同项下的多次分批采购和付款安排。"
    },
    {
      "title": "特殊下单场景",
      "definition": "用户需要处理超出常规流程的订单需求，包括紧急订单处理、特殊客户定制需求、异常订单修改、优先级调整、特殊折扣申请或跨部门协调的复杂订单场景。"
    },
    {
      "title": "收货与验收",
      "definition": "用户希望通过人工智能助手协助完成货物接收和项目交付的验收工作，包括生成收货清单、验收报告、质量检查记录，以及跟踪验收流程和处理验收异常情况。"
    },
    {
      "title": "供应商入库",
      "definition": "用户的核心目的是将新的供应商信息录入到企业管理系统中，包括供应商基本信息、资质证明、联系方式、业务范围、合作条款等关键数据的标准化录入和审核流程管理。"
    },
    {
      "title": "供应商信息管理",
      "definition": "用户希望对企业的供应商资源进行全面管理，包括供应商基本信息录入、资质审核、绩效评估、合同管理、付款跟踪等，以及查询供应商详细信息、生成供应商评估报告、维护供应商数据库等操作。"
    },
    {
      "title": "采购招投标管理",
      "definition": "用户希望通过人工智能助手来协助处理采购招投标流程中的各项工作，包括制定招标文件、供应商资质审查、投标文件评审、比价分析、合同条款梳理、供应商管理以及采购决策支持等，以提高采购效率和决策质量。"
    },
    {
      "title": "报价处理与审批",
      "definition": "用户希望人工智能协助处理商务报价流程中的各个环节，包括根据客户需求生成标准化报价单、对比分析多个供应商报价、跟踪报价状态、协助完成报价审批流程，以及生成报价分析报告等相关文档。"
    },
    {
      "title": "智能寻源管理",
      "definition": "用户希望人工智能协助进行供应商发现、评估和采购决策，包括供应商信息收集、价格比较分析、供应商资质评估、采购方案推荐、合同条款建议等，以提高采购效率和降低成本风险。"
    },
    {
      "title": "采购系统操作指引",
      "definition": "用户希望获得采购相关系统的使用帮助和操作指导，包括采购流程步骤、系统功能介绍、操作权限说明、审批流程指引、供应商管理操作等，以提高采购工作效率和规范性。"
    },
    {
      "title": "采购系统异常处理",
      "definition": "用户在使用采购系统过程中遇到技术故障、流程卡顿、数据错误或功能异常时，需要快速识别问题原因、获取解决方案指导，或协助进行系统恢复和异常情况的应急处理。"
    },
    {
      "title": "采购权限管理",
      "definition": "用户希望管理和控制采购流程中的权限分配，包括查询采购审批权限、设置采购金额限制、分配供应商管理权限、控制采购订单创建和修改权限，以及生成采购权限报告等，确保采购活动符合公司内控制度和合规要求。"
    },
    {
      "title": "采购培训与认证",
      "definition": "用户希望获取采购相关的培训资源、认证信息或学习指导，包括采购流程培训、供应商管理认证、采购法规学习、采购技能提升课程推荐，以及相关认证考试的准备和申请流程。"
    }
  ]
"""

with open(BASE_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", work_function_description)

print(f"正在生成 {DOMAIN} Tools...")
print("-" * 60)
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 purchase Tools...
------------------------------------------------------------


```python
"""Toolkit for the purchase/procurement domain, extending OfficeTools."""
from typing import Any, Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolType, is_tool
from bench.domains.base.tools import OfficeTools
from bench.domains.purchase.models import (
    AcceptanceItem,
    AcceptanceRecord,
    ApprovalAction,
    ApprovalFlowNode,
    ApprovalFlowTemplate,
    ApprovalInstance,
    ContractItem,
    GoodsReceipt,
    GoodsReceiptItem,
    ProcurementCategory,
    ProcurementIssueTicket,
    ProcurementPermissionRule,
    ProcurementTrainingCourse,
    ProcurementTrainingRecord,
    PurchaseChangeRequest,
    PurchaseContract,
    PurchaseDB,
    PurchaseOrder,
    PurchaseOrderItem,
    PurchaseRequisition,
    Quotation,
    QuotationItem,
    RequisitionItem,
    SourcingProject,
    SourcingSupplier,
    Supplier,
   